In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("hand_landmarks_data.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25675 entries, 0 to 25674
Data columns (total 64 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x1      25675 non-null  float64
 1   y1      25675 non-null  float64
 2   z1      25675 non-null  float64
 3   x2      25675 non-null  float64
 4   y2      25675 non-null  float64
 5   z2      25675 non-null  float64
 6   x3      25675 non-null  float64
 7   y3      25675 non-null  float64
 8   z3      25675 non-null  float64
 9   x4      25675 non-null  float64
 10  y4      25675 non-null  float64
 11  z4      25675 non-null  float64
 12  x5      25675 non-null  float64
 13  y5      25675 non-null  float64
 14  z5      25675 non-null  float64
 15  x6      25675 non-null  float64
 16  y6      25675 non-null  float64
 17  z6      25675 non-null  float64
 18  x7      25675 non-null  float64
 19  y7      25675 non-null  float64
 20  z7      25675 non-null  float64
 21  x8      25675 non-null  float64
 22

In [4]:
df.head()

,x1,y1,z1,x2,y2,z2,x3,y3,z3,x4,...,x19,y19,z19,x20,y20,z20,x21,y21,z21,label
0,262.669968,257.304901,-3.649205e-07,257.417542,247.109055,0.004224,246.882957,241.716827,0.005798,236.384537,...,223.345093,255.490692,-0.020450,215.043365,258.114746,-0.024577,208.006393,259.608673,-0.026722,call
1,83.351778,346.059113,-2.345265e-07,81.925037,328.562347,-0.011102,90.080132,311.535248,-0.021096,95.641823,...,132.451618,341.794434,-0.038175,142.773582,342.829254,-0.037336,152.431698,343.015991,-0.036136,call
2,187.756977,260.235492,-2.417307e-07,195.460579,241.506035,-0.000184,207.259529,223.674339,-0.009687,215.413628,...,250.301010,268.602938,-0.044068,262.425133,271.276638,-0.040469,272.989952,272.272231,-0.038301,call
3,114.976696,331.594238,-1.233261e-07,114.503494,320.549957,-0.002824,116.636627,310.080994,-0.008911,117.685066,...,145.195450,329.357544,-0.027622,151.053200,329.712341,-0.027863,155.990364,329.548828,-0.027723,call
4,188.795288,141.727867,-1.622995e-07,188.520905,127.947464,-0.002884,191.982880,111.010563,-0.008115,192.552521,...,226.696396,132.263248,-0.025113,234.831741,130.684147,-0.024087,241.587769,128.477188,-0.023486,call


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INPUT_SIZE = 63      # 21 landmarks * (x,y,z)
NUM_CLASSES = 18
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

# =========================================================
# Dataset
# =========================================================

class LandmarkDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================================================
# Simplest Neural Network (1 hidden layer)
# =========================================================

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(INPUT_SIZE, 128),
            nn.ReLU(),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        return self.model(x)


# =========================================================
# Load Data
# =========================================================



X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_ds = LandmarkDataset(X_train, y_train)
val_ds = LandmarkDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# =========================================================
# Training
# =========================================================

model = SimpleMLP().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):

    # ---- train ----
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # ---- validation ----
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            outputs = model(X_batch)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            targets.extend(y_batch.numpy())

    acc = accuracy_score(targets, preds)
    print(f"Epoch {epoch+1}/{EPOCHS} - Val Accuracy: {acc:.4f}")

print("Training Finished")

Epoch 1/50 - Val Accuracy: 0.3949
Epoch 2/50 - Val Accuracy: 0.5755
Epoch 3/50 - Val Accuracy: 0.6070
Epoch 4/50 - Val Accuracy: 0.6874
Epoch 5/50 - Val Accuracy: 0.7352
Epoch 6/50 - Val Accuracy: 0.7328
Epoch 7/50 - Val Accuracy: 0.7747
Epoch 8/50 - Val Accuracy: 0.8152
Epoch 9/50 - Val Accuracy: 0.8103
Epoch 10/50 - Val Accuracy: 0.8113
Epoch 11/50 - Val Accuracy: 0.7706
Epoch 12/50 - Val Accuracy: 0.8682
Epoch 13/50 - Val Accuracy: 0.8835
Epoch 14/50 - Val Accuracy: 0.8798
Epoch 15/50 - Val Accuracy: 0.8101
Epoch 16/50 - Val Accuracy: 0.8339
Epoch 17/50 - Val Accuracy: 0.8676
Epoch 18/50 - Val Accuracy: 0.8849
Epoch 19/50 - Val Accuracy: 0.8454
Epoch 20/50 - Val Accuracy: 0.9242
Epoch 21/50 - Val Accuracy: 0.9204
Epoch 22/50 - Val Accuracy: 0.8943
Epoch 23/50 - Val Accuracy: 0.8405
Epoch 24/50 - Val Accuracy: 0.8351
Epoch 25/50 - Val Accuracy: 0.8974
Epoch 26/50 - Val Accuracy: 0.8851
Epoch 27/50 - Val Accuracy: 0.9061
Epoch 28/50 - Val Accuracy: 0.9287
Epoch 29/50 - Val Accuracy: 0

In [13]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(targets, preds)

print(f"\nValidation Accuracy: {acc:.4f}")
print("\nClassification Report:\n")
print(classification_report(targets, preds))


Validation Accuracy: 0.9373

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.97      0.96       301
           1       1.00      1.00      1.00       259
           2       0.99      0.97      0.98       189
           3       0.95      0.98      0.96       327
           4       0.98      0.94      0.96       287
           5       0.88      0.84      0.86       217
           6       1.00      0.98      0.99       318
           7       0.85      0.89      0.87       253
           8       0.96      0.98      0.97       330
           9       0.90      0.82      0.86       288
          10       0.96      0.88      0.92       299
          11       0.99      0.99      0.99       292
          12       0.85      0.97      0.91       296
          13       0.97      0.90      0.94       314
          14       0.98      0.96      0.97       291
          15       0.99      0.96      0.98       331
          16       0.77    

Deep MLP

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_SIZE = 63
NUM_CLASSES = 18
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3


class DeepMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(INPUT_SIZE, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x):
        return self.model(x)


model = DeepMLP().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):

    # ---- Train ----
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ---- Validation ----
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            outputs = model(X_batch)

            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            targets.extend(y_batch.numpy())

    acc = accuracy_score(targets, preds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | Val Acc: {acc:.4f}")

print("Training Finished")

Epoch 1/60 | Loss: 724.8741 | Val Acc: 0.5807
Epoch 2/60 | Loss: 322.8123 | Val Acc: 0.5646
Epoch 3/60 | Loss: 255.9394 | Val Acc: 0.7174
Epoch 4/60 | Loss: 209.7704 | Val Acc: 0.7420
Epoch 5/60 | Loss: 167.1732 | Val Acc: 0.7745
Epoch 6/60 | Loss: 164.8566 | Val Acc: 0.8051
Epoch 7/60 | Loss: 134.3534 | Val Acc: 0.9059
Epoch 8/60 | Loss: 123.9775 | Val Acc: 0.8162
Epoch 9/60 | Loss: 110.9500 | Val Acc: 0.9334
Epoch 10/60 | Loss: 103.6693 | Val Acc: 0.9069
Epoch 11/60 | Loss: 97.2619 | Val Acc: 0.8911
Epoch 12/60 | Loss: 95.1855 | Val Acc: 0.9352
Epoch 13/60 | Loss: 81.7243 | Val Acc: 0.8031
Epoch 14/60 | Loss: 85.9426 | Val Acc: 0.9357
Epoch 15/60 | Loss: 78.8959 | Val Acc: 0.9168
Epoch 16/60 | Loss: 82.6825 | Val Acc: 0.9083
Epoch 17/60 | Loss: 82.6752 | Val Acc: 0.9155
Epoch 18/60 | Loss: 74.2318 | Val Acc: 0.9262
Epoch 19/60 | Loss: 63.8542 | Val Acc: 0.9496
Epoch 20/60 | Loss: 65.2245 | Val Acc: 0.9507
Epoch 21/60 | Loss: 75.7393 | Val Acc: 0.9311
Epoch 22/60 | Loss: 55.4870 | Val

Applying best practices techniques

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import optuna
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import copy
import random

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 18
INPUT_SIZE = 63  # 21 landmarks * (x,y,z)

# =========================================================
# 1. Dataset + Augmentation
# =========================================================

class LandmarkDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def _augment(self, x):
        x = x.view(21, 3)

        # random gaussian noise
        x += torch.randn_like(x) * 0.01

        # random scaling
        scale = random.uniform(0.9, 1.1)
        x *= scale

        # random rotation around z-axis
        theta = random.uniform(-0.2, 0.2)
        rot = torch.tensor([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta),  np.cos(theta), 0],
            [0, 0, 1]
        ], dtype=torch.float32)
        x = torch.matmul(x, rot)

        return x.view(-1)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]

        if self.augment:
            x = self._augment(x)

        return x, y


# =========================================================
# 2. Residual Block (MLP-ResNet style)
# =========================================================

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class ResNetMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_blocks, dropout):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )

        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )

        self.output_layer = nn.Linear(hidden_dim, NUM_CLASSES)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        return self.output_layer(x)


# =========================================================
# 3. Early Stopping
# =========================================================

class EarlyStopping:
    def __init__(self, patience=15):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0
        self.best_model = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1

        return self.counter >= self.patience


# =========================================================
# 4. Training Function
# =========================================================

def train_model(model, train_loader, val_loader, lr, weight_decay):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

    early_stopping = EarlyStopping(patience=20)

    for epoch in range(200):
        model.train()
        train_loss = 0

        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0
        preds, targets = [], []

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                outputs = model(X)
                loss = criterion(outputs, y)
                val_loss += loss.item()

                preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                targets.extend(y.cpu().numpy())

        val_acc = accuracy_score(targets, preds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch} | Val Acc: {val_acc:.4f}")

        if early_stopping.step(val_loss, model):
            print("Early stopping")
            break

    model.load_state_dict(early_stopping.best_model)
    return model


# =========================================================
# 5. Optuna Optimization
# =========================================================

def objective(trial):

    hidden_dim = trial.suggest_int("hidden_dim", 128, 512)
    num_blocks = trial.suggest_int("num_blocks", 2, 6)
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    model = ResNetMLP(INPUT_SIZE, hidden_dim, num_blocks, dropout).to(DEVICE)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    model = train_model(model, train_loader, val_loader, lr, weight_decay)

    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            outputs = model(X)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            targets.extend(y.cpu().numpy())

    return accuracy_score(targets, preds)


# =========================================================
# 6. Load Data
# =========================================================

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_ds = LandmarkDataset(X_train, y_train, augment=True)
val_ds = LandmarkDataset(X_val, y_val, augment=False)

# =========================================================
# 7. Run Bayesian Optimization
# =========================================================

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Accuracy:", study.best_value)

[I 2026-02-27 20:11:28,143] A new study created in memory with name: no-name-4682923e-7836-4701-92d6-546d2169bed0
/tmp/ipython-input-993/911607322.py:191: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
/tmp/ipython-input-993/911607322.py:192: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-2)


Epoch 0 | Val Acc: 0.4816
Epoch 1 | Val Acc: 0.6717
Epoch 2 | Val Acc: 0.7671
Epoch 3 | Val Acc: 0.7472
Epoch 4 | Val Acc: 0.5986
Epoch 5 | Val Acc: 0.7217
Epoch 6 | Val Acc: 0.6189
Epoch 7 | Val Acc: 0.8483
Epoch 8 | Val Acc: 0.8970
Epoch 9 | Val Acc: 0.8726
Epoch 10 | Val Acc: 0.8107
Epoch 11 | Val Acc: 0.8995
Epoch 12 | Val Acc: 0.8312
Epoch 13 | Val Acc: 0.9235
Epoch 14 | Val Acc: 0.7805
Epoch 15 | Val Acc: 0.8927
Epoch 16 | Val Acc: 0.7305
Epoch 17 | Val Acc: 0.8948
Epoch 18 | Val Acc: 0.8284
Epoch 19 | Val Acc: 0.8958
Epoch 20 | Val Acc: 0.9840
Epoch 21 | Val Acc: 0.9844
Epoch 22 | Val Acc: 0.9823
Epoch 23 | Val Acc: 0.9842
Epoch 24 | Val Acc: 0.9827
Epoch 25 | Val Acc: 0.9846
Epoch 26 | Val Acc: 0.9815
Epoch 27 | Val Acc: 0.9823
Epoch 28 | Val Acc: 0.9815
Epoch 29 | Val Acc: 0.9852
Epoch 30 | Val Acc: 0.9776
Epoch 31 | Val Acc: 0.9831
Epoch 32 | Val Acc: 0.9780
Epoch 33 | Val Acc: 0.9846
Epoch 34 | Val Acc: 0.9838
Epoch 35 | Val Acc: 0.9827
Epoch 36 | Val Acc: 0.9866
Epoch 37 | 

In [17]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.7 MB/s eta 0:00:00
